# Volt Notebook

Use Python to interact with your simulations, analyses, and results on the Volt platform.
This notebook is pre-configured to connect to your Volt instance.

## Setup

In [ ]:
from voltsdk import VoltClient
import os

client = VoltClient.from_env()
print(f"Connected to team: {client.team.name}")

## Load Trajectory

In [ ]:
trajectory_id = os.environ.get('VOLT_TRAJECTORY_ID', '')

if trajectory_id:
    traj = client.trajectories.get(trajectory_id)
    print(f"Trajectory: {traj.name}")
    print(f"Status: {traj.status}")
    print(f"Frames: {traj.frame_count}")
else:
    print("No trajectory linked to this notebook.")
    print("\nAvailable trajectories:")
    for t in client.trajectories.list():
        print(f"  {t.name} (id={t.id}, {t.frame_count} frames)")
    print("\nTo load one: traj = client.trajectories.get('paste_id_here')")

## List Analyses

In [ ]:
if trajectory_id:
    for analysis in traj.analyses:
        progress = f"{analysis.progress:.0%}" if analysis.status == 'completed' else analysis.status
        print(f"  {analysis.plugin_name}: {progress}")
        print(f"    ID: {analysis.id}")
        for key, val in analysis.config.items():
            print(f"    {key}: {val}")
        print()

## Explore Listing Data

Each analysis produces listing rows (tabular data indexed by timestep).
These can be loaded directly into pandas DataFrames for analysis.

In [ ]:
if trajectory_id:
    analysis = traj.analyses.first()
    if analysis:
        df = analysis.listings.to_dataframe()
        print(f"Plugin: {analysis.plugin_name}")
        print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
        print()
        df.head(10)
    else:
        print("No analyses yet. Run an analysis plugin on this trajectory in Volt.")

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt

if trajectory_id and analysis and not df.empty:
    # Auto-detect numeric columns for plotting
    exclude = {'_id', 'analysisId', 'trajectoryId', 'exposureId', 'trajectoryName'}
    numeric_cols = [c for c in df.select_dtypes(include='number').columns
                    if c not in exclude and c != 'timestep']

    if 'timestep' in df.columns and numeric_cols:
        # Plot up to 4 columns
        cols_to_plot = numeric_cols[:4]
        n = len(cols_to_plot)
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
        if n == 1:
            axes = [axes]

        for ax, col in zip(axes, cols_to_plot):
            df.plot(x='timestep', y=col, ax=ax, legend=False)
            ax.set_title(col)
            ax.set_ylabel(col)

        plt.tight_layout()
        plt.show()
    else:
        print("No numeric timeseries data to plot.")
        print(f"Available columns: {list(df.columns)}")

## Download Analysis Artifacts

Download the raw analysis output files (MessagePack data + GLB 3D models)
for deeper analysis or use with external tools.

In [ ]:
# Uncomment to download artifacts for the first analysis:
# if analysis:
#     output_dir = analysis.download_artifacts(dest='./analysis_data/')
#     print(f"Artifacts saved to: {output_dir}")

## Load MsgPack Results into DataFrames

After downloading artifacts, you can load the raw MessagePack files
into DataFrames for full access to all analysis output data.

In [ ]:
# Uncomment after downloading artifacts:
# from voltsdk import msgpack_as_df
#
# df_raw = msgpack_as_df(
#     file_path='./analysis_data/.../timestep-0.msgpack.zst',
#     iterable_key='data'
# )
# df_raw.head()

## View 3D Models

In [ ]:
# Uncomment to view a GLB model inline:
# from voltsdk import view_glb
# view_glb(file_path='./analysis_data/.../model.glb.zst')

## Quick Reference

| Task | Code |
|------|------|
| Connect | `client = VoltClient.from_env()` |
| List trajectories | `client.trajectories.list()` |
| Get trajectory | `client.trajectories.get("id")` |
| List analyses | `traj.analyses.list()` |
| Listings to DataFrame | `analysis.listings.to_dataframe()` |
| Export to CSV | `analysis.listings.to_csv("output.csv")` |
| Quick plot | `traj.plot("temperature")` |
| Plot listing | `analysis.listings.plot(x="timestep", y="energy")` |
| Download artifacts | `analysis.download_artifacts(dest="./data/")` |
| Atom data | `frame.atoms()` |
| Download dump | `frame.download_dump()` |
| Download GLB | `frame.download_glb()` |
| OVITO pipeline | `traj.to_ovito_pipeline()` |
| MsgPack to DataFrame | `msgpack_as_df("file.msgpack.zst", iterable_key="data")` |
| View 3D model | `view_glb("model.glb.zst")` |